# Candidate recovery metric: post-operative length of stay

Length of stay is the highest-coverage recovery proxy available, because it is derived from
records every admitted patient generates rather than from a device they chose to wear. Its
weaknesses are the opposite of the wearables': it is coarse, it is partly administrative rather
than clinical, and it overlaps with the post-operative events a model might otherwise use as
features.

The audit questions are therefore whether it has coverage (does the index surgery sit inside an
identifiable inpatient stay), whether it has spread (a near-constant outcome cannot be
predicted), and how much leakage it carries.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("length_of_stay")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = srp_common.index_visit_cte(cfg)

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# 2-coverage

### How many patients' index surgeries sit inside an identifiable inpatient stay?

A day-case surgery has no stay to measure, and a surgery whose visit record is missing or
undated is invisible to this metric. The gap between the cohort and this count is the population
a length-of-stay outcome silently drops.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                                   AS cohort_patients,
        (SELECT COUNT(*) FROM index_surgery)                            AS patients_with_index_date,
        (SELECT COUNT(*) FROM index_visit WHERE rn = 1)                 AS patients_with_index_visit,
        (SELECT COUNTIF(rn > 1) FROM index_visit)                       AS extra_overlapping_visits,
        (SELECT COUNTIF(postop_los_days < 0) FROM index_visit WHERE rn = 1)
                                                                        AS negative_postop_los
"""

cov = cached_gbq(coverage_sql, "length_of_stay_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])
patients_with_index_visit = int(cov["patients_with_index_visit"])

print(f"cohort patients                    : {cohort_patients:>9,}")
print(f"  ... with an index date           : {int(cov['patients_with_index_date']):>9,}")
print(f"  ... with an inpatient index stay : {patients_with_index_visit:>9,}"
      f"   ({patients_with_index_visit / cohort_patients:6.1%} of cohort)")
print(f"overlapping stays discarded by rn  : {int(cov['extra_overlapping_visits']):>9,}")
print(f"negative post-op LOS (data errors) : {int(cov['negative_postop_los']):>9,}")

# 3-distribution

### What does post-operative length of stay look like?

Stays of 30 days or more are pooled into one bucket, since the tail is long and individually
sparse. The suppressed variant additionally aggregates any bucket below the threshold.

In [ ]:
los_sql = CTES + """
    SELECT
        LEAST(postop_los_days, 30) AS los_bucket,
        COUNT(*)                   AS n_patients
    FROM index_visit
    WHERE rn = 1
      AND postop_los_days >= 0
    GROUP BY los_bucket
    ORDER BY los_bucket
"""

los_dist = cached_gbq(los_sql, "length_of_stay_distribution")
los_dist["los_label"] = [
    "30+" if int(v) >= 30 else str(int(v)) for v in los_dist["los_bucket"]
]

srp_common.plot_distribution(
    cfg, los_dist, "los_label",
    title="Patients by post-operative length of stay",
    xlabel="days from index surgery to discharge", suppress=False,
    save_as="length_of_stay")
srp_common.plot_distribution(
    cfg, los_dist, "los_label",
    title="Patients by post-operative length of stay",
    xlabel="days from index surgery to discharge", suppress=True,
    save_as="length_of_stay")

### Does it have enough spread to be worth predicting?

An outcome concentrated on one or two values carries almost no information, whatever its
coverage. The share sitting on the modal value is the quickest read on that.

In [ ]:
quantile_sql = CTES + """
    SELECT
        COUNT(*)                                                AS n_patients,
        APPROX_QUANTILES(postop_los_days, 100)[OFFSET(10)]      AS p10,
        APPROX_QUANTILES(postop_los_days, 100)[OFFSET(25)]      AS p25,
        APPROX_QUANTILES(postop_los_days, 100)[OFFSET(50)]      AS p50,
        APPROX_QUANTILES(postop_los_days, 100)[OFFSET(75)]      AS p75,
        APPROX_QUANTILES(postop_los_days, 100)[OFFSET(90)]      AS p90,
        AVG(postop_los_days)                                    AS mean_los,
        STDDEV(postop_los_days)                                 AS sd_los
    FROM index_visit
    WHERE rn = 1 AND postop_los_days >= 0
"""

los_stats = cached_gbq(quantile_sql, "length_of_stay_quantiles").iloc[0]
modal_row = los_dist.loc[los_dist["n_patients"].idxmax()]
modal_share = int(modal_row["n_patients"]) / int(los_stats["n_patients"])

print("post-op LOS (days)  p10={p10}  p25={p25}  median={p50}  p75={p75}  p90={p90}".format(
    **{k: int(los_stats[k]) for k in ["p10", "p25", "p50", "p75", "p90"]}))
print(f"mean={los_stats['mean_los']:.2f}  sd={los_stats['sd_los']:.2f}  "
      f"IQR={int(los_stats['p75']) - int(los_stats['p25'])}")
print(f"modal value = {modal_row['los_label']} days, holding "
      f"{int(modal_row['n_patients']):,} patients ({modal_share:.1%})")
print()
print("--> A modal share above roughly half would make this close to a constant, "
      "and a poor prediction target however well covered it is.")

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="ehr",
    source_table="visit_occurrence",
    index_rule=INDEX_RULE,
    outcome_type="count",
    cohort_patients=cohort_patients,
    patients_with_metric=patients_with_index_visit,
    patients_analysable=int(los_stats["n_patients"]),
    coverage_share=int(los_stats["n_patients"]) / cohort_patients,
    horizon_days=None,
    median_value=float(los_stats["p50"]),
    iqr=float(los_stats["p75"] - los_stats["p25"]),
    modal_share=float(modal_share),
    informative_missingness="no",
    leakage_risk="high",
    notes=("Leakage is the main objection: LOS is partly determined by the post-operative "
           "complications a model would otherwise want as features, and partly by discharge "
           "administration rather than the patient's recovery."),
)